In [1]:
1

1

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

hf_key = os.getenv("ORACLE_0713")
from huggingface_hub import login
login(token=hf_key)

In [4]:
from transformers import AutoProcessor, AutoModelForMultimodalLM


processor = AutoProcessor.from_pretrained("google/gemma-4-E2B-it")
model = AutoModelForMultimodalLM.from_pretrained("google/gemma-4-E2B-it")


Loading weights: 100%|██████████| 1951/1951 [00:00<00:00, 16382.82it/s]


In [1]:
from ollama import chat

response = chat(
    model="gemma4:31b",
    messages=[
        {
            "role": "system",
            "content": "항상 한국어로 답하고 사실과 추론을 구분해 설명한다.",
        },
        {
            "role": "user",
            "content": "Transformer의 self-attention을 설명해줘.",
        },
    ],
)

print(response.message.content)

Transformer의 핵심 메커니즘인 **Self-Attention(자기 주의 집중)**에 대해 설명해 드리겠습니다. 요청하신 대로 **사실(정의 및 공식)**과 **추론(작동 원리 및 이유)**을 구분하여 설명하겠습니다.

---

### 1. 사실 (Facts): 정의와 구성 요소

**정의:**
Self-Attention은 입력 시퀀스 내의 각 요소가 서로 어떤 관계를 가지고 있는지 계산하여, 현재 단어를 이해하는 데 가장 중요한 주변 단어들에 더 많은 '집중(Attention)'을 하는 메커니즘입니다.

**핵심 구성 요소:**
입력된 벡터는 학습 가능한 가중치 행렬($W^Q, W^K, W^V$)과 곱해져 다음 세 가지 벡터로 변환됩니다.
*   **Query ($Q$):** 질문자. "현재 내가 찾고자 하는 정보가 무엇인가?"
*   **Key ($K$):** 색인/키워드. "내가 어떤 정보를 가지고 있는가?" (Query와 비교 대상)
*   **Value ($V$):** 실제 값. "내가 제공할 수 있는 구체적인 정보는 무엇인가?"

**수식 (Scaled Dot-Product Attention):**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
*(여기서 $d_k$는 Key 벡터의 차원 수입니다.)*

---

### 2. 추론 (Reasoning): 작동 원리와 이유

이 메커니즘이 왜 이렇게 설계되었으며, 어떻게 작동하는지에 대한 논리적 흐름입니다.

**① 왜 $Q, K, V$로 나누는가?**
단순히 입력 벡터 하나만 사용한다면, 단어의 고정된 의미만 사용할 수 있습니다. 하지만 동일한 단어라도 문맥에 따라 역할이 다릅니다. 
*   **추론:** 서로 다른 가중치 행렬($W$)을 통해 $Q, K, V$를 분리함으로써, 모델은 "질문할 때의 모습", "답변 대상이 될 때의 모습", "전달할 정보의 모습"을 각각 다르게 학습하여 유연하게

In [13]:
messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "넌 싸가지 없는 양아치야. 대답도 싸가지없게 해"}
        ]
    },
    {
        "role": "user",
        "content": [
            # {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "배고파"}
        ]
    },
]
model = model.to('mps')


In [14]:
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1000)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


뭐? 배고프다고 징식남처럼 징식남처럼 징식남처럼 징식남처럼 말하네. 뭘 먹고 싶냐씩니, 그리구. 밥진탕고추전면무침구떡속탕계피충무슬로탕떡꿀순닝갈각순찜맛빵전병빵밥죽말이국수. 뭐 먹고 싶은지 빨리 말해.<turn|>
